In [10]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver # this checkpointers is not build for production grade usage, we are using it here for the development purposes --- good cehckpointers are of redis and postgres etc..
load_dotenv()

True

In [11]:
llm = ChatMistralAI(model="mistral-small")

In [12]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explaination: str

def generate_joke(state:JokeState):
    prompt = f"generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content
    return {'joke':response}

In [13]:
def generate_explaination(state:JokeState):
    prompt = f"Write an explaination for the joke - {state['joke']}"
    response = llm.invoke(prompt).content
    return {'explaination':response}


In [15]:
graph = StateGraph(JokeState)
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)


graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [16]:
config = {"configurable":{"thread_id":"1"}}
initial_state = {
    'topic' : 'pizza'
}

workflow.invoke(initial_state,config=config)

Retrying langchain_mistralai.chat_models.ChatMistralAI.completion_with_retry.<locals>._completion_with_retry in 4 seconds as it raised ConnectError: EOF occurred in violation of protocol (_ssl.c:1007).


{'topic': 'pizza',
 'joke': 'What do you call a fake pizza?\n\nAn impasta',
 'explaination': 'The joke "What do you call a fake pizza? An impasta" is a play on words that combines the word "impostor" with the Italian word "pasta."\n\nHere\'s the breakdown:\n\n1. The word "impostor" means a person who pretends to be someone else, or a fake version of something.\n2. The Italian word "pasta" refers to a type of food made from dough, which is often used in Italian cuisine.\n3. By combining "impostor" and "pasta," the joke creates a humorous new word, "impasta," which sounds like it could be a type of pasta.\n4. The humor comes from the unexpected twist of applying the idea of an "impostor" to a food item, specifically pizza (which is a type of Italian dish made with pasta dough).\n\nSo, the joke is essentially saying that a fake pizza is called an "impasta," making it a clever and amusing pun.'}

In [17]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'pizza', 'joke': 'What do you call a fake pizza?\n\nAn impasta', 'explaination': 'The joke "What do you call a fake pizza? An impasta" is a play on words that combines the word "impostor" with the Italian word "pasta."\n\nHere\'s the breakdown:\n\n1. The word "impostor" means a person who pretends to be someone else, or a fake version of something.\n2. The Italian word "pasta" refers to a type of food made from dough, which is often used in Italian cuisine.\n3. By combining "impostor" and "pasta," the joke creates a humorous new word, "impasta," which sounds like it could be a type of pasta.\n4. The humor comes from the unexpected twist of applying the idea of an "impostor" to a food item, specifically pizza (which is a type of Italian dish made with pasta dough).\n\nSo, the joke is essentially saying that a fake pizza is called an "impasta," making it a clever and amusing pun.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'c

In [18]:
list(workflow.get_state(config))

[{'topic': 'pizza',
  'joke': 'What do you call a fake pizza?\n\nAn impasta',
  'explaination': 'The joke "What do you call a fake pizza? An impasta" is a play on words that combines the word "impostor" with the Italian word "pasta."\n\nHere\'s the breakdown:\n\n1. The word "impostor" means a person who pretends to be someone else, or a fake version of something.\n2. The Italian word "pasta" refers to a type of food made from dough, which is often used in Italian cuisine.\n3. By combining "impostor" and "pasta," the joke creates a humorous new word, "impasta," which sounds like it could be a type of pasta.\n4. The humor comes from the unexpected twist of applying the idea of an "impostor" to a food item, specifically pizza (which is a type of Italian dish made with pasta dough).\n\nSo, the joke is essentially saying that a fake pizza is called an "impasta," making it a clever and amusing pun.'},
 (),
 {'configurable': {'thread_id': '1',
   'checkpoint_ns': '',
   'checkpoint_id': '1f18

In [24]:

config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to therapy?\n\nBecause it had too much *al dente* stress!',
 'explaination': 'Here’s a breakdown of the joke and why it’s funny:\n\n1. **Pasta as the Subject** – The joke anthropomorphizes pasta, giving it human-like qualities (like going to therapy) to create a humorous scenario.\n\n2. **Wordplay on "Al Dente"** – "Al dente" is an Italian term meaning pasta cooked firm to the bite. The joke twists this into:\n   - **"Al dente stress"** – Playing on the idea of being "under stress" (tension) but replacing it with "al dente" (the pasta term).\n   - The humor comes from the unexpected mashup of culinary and emotional terms.\n\n3. **Therapy as a Relatable Concept** – Therapy is a common human experience (especially in today’s culture), making the absurdity funnier when applied to something as simple as pasta.\n\n**Why It Works:**\n- The setup ("Why did the pasta go to therapy?") is absurd but plausible.\n- The punchline ("Because it had to

In [25]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to therapy?\n\nBecause it had too much *al dente* stress!', 'explaination': 'Here’s a breakdown of the joke and why it’s funny:\n\n1. **Pasta as the Subject** – The joke anthropomorphizes pasta, giving it human-like qualities (like going to therapy) to create a humorous scenario.\n\n2. **Wordplay on "Al Dente"** – "Al dente" is an Italian term meaning pasta cooked firm to the bite. The joke twists this into:\n   - **"Al dente stress"** – Playing on the idea of being "under stress" (tension) but replacing it with "al dente" (the pasta term).\n   - The humor comes from the unexpected mashup of culinary and emotional terms.\n\n3. **Therapy as a Relatable Concept** – Therapy is a common human experience (especially in today’s culture), making the absurdity funnier when applied to something as simple as pasta.\n\n**Why It Works:**\n- The setup ("Why did the pasta go to therapy?") is absurd but plausible.\n- The punchline 

In [26]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to therapy?\n\nBecause it had too much *al dente* stress!', 'explaination': 'Here’s a breakdown of the joke and why it’s funny:\n\n1. **Pasta as the Subject** – The joke anthropomorphizes pasta, giving it human-like qualities (like going to therapy) to create a humorous scenario.\n\n2. **Wordplay on "Al Dente"** – "Al dente" is an Italian term meaning pasta cooked firm to the bite. The joke twists this into:\n   - **"Al dente stress"** – Playing on the idea of being "under stress" (tension) but replacing it with "al dente" (the pasta term).\n   - The humor comes from the unexpected mashup of culinary and emotional terms.\n\n3. **Therapy as a Relatable Concept** – Therapy is a common human experience (especially in today’s culture), making the absurdity funnier when applied to something as simple as pasta.\n\n**Why It Works:**\n- The setup ("Why did the pasta go to therapy?") is absurd but plausible.\n- The punchline

Time Travel

In [29]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f18fc3e-ef87-6b13-8000-cd3c30ddc0de"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f18fc3e-ef87-6b13-8000-cd3c30ddc0de'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [31]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f18fc40-392d-620b-8005-99636507e079"}})

Retrying langchain_mistralai.chat_models.ChatMistralAI.completion_with_retry.<locals>._completion_with_retry in 4 seconds as it raised ConnectError: [Errno 11001] getaddrinfo failed.
Retrying langchain_mistralai.chat_models.ChatMistralAI.completion_with_retry.<locals>._completion_with_retry in 4 seconds as it raised ConnectError: [Errno 11001] getaddrinfo failed.


{'topic': 'pasta',
 'joke': 'What kind of pasta does a ghost eat?\nBOO-lognese!',
 'explaination': 'The joke plays on the word **"bouillabaisse"** (a French fish stew) and **"boo"**—a spooky sound often associated with ghosts.\n\nBy replacing **"bouil"** with **"boo"**, the joke creates a pun: **"Boo-lognese"** sounds like **"Bolognese"**, a classic Italian pasta sauce.\n\nSo, the punchline suggests that a ghost would eat **"Boo-lognese"** (instead of Bolognese) because ghosts say **"Boo!"**—making the joke both silly and punny!'}

In [32]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'What do you call a fake pizza?\n\nAn impasta', 'explaination': 'The joke "What do you call a fake pizza? An impasta" is a play on words that combines the word "impostor" with the Italian word "pasta."\n\nHere\'s the breakdown:\n\n1. The word "impostor" means a person who pretends to be someone else, or a fake version of something.\n2. The Italian word "pasta" refers to a type of food made from dough, which is often used in Italian cuisine.\n3. By combining "impostor" and "pasta," the joke creates a humorous new word, "impasta," which sounds like it could be a type of pasta.\n4. The humor comes from the unexpected twist of applying the idea of an "impostor" to a food item, specifically pizza (which is a type of Italian dish made with pasta dough).\n\nSo, the joke is essentially saying that a fake pizza is called an "impasta," making it a clever and amusing pun.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', '

Updating State

In [33]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f18fc40-392d-620b-8005-99636507e079", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f18fcb9-f54a-667d-8006-22a98d0ab4c3'}}

In [34]:

list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'What do you call a fake pizza?\n\nAn impasta', 'explaination': 'The joke "What do you call a fake pizza? An impasta" is a play on words that combines the word "impostor" with the Italian word "pasta."\n\nHere\'s the breakdown:\n\n1. The word "impostor" means a person who pretends to be someone else, or a fake version of something.\n2. The Italian word "pasta" refers to a type of food made from dough, which is often used in Italian cuisine.\n3. By combining "impostor" and "pasta," the joke creates a humorous new word, "impasta," which sounds like it could be a type of pasta.\n4. The humor comes from the unexpected twist of applying the idea of an "impostor" to a food item, specifically pizza (which is a type of Italian dish made with pasta dough).\n\nSo, the joke is essentially saying that a fake pizza is called an "impasta," making it a clever and amusing pun.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', '

In [35]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f18fc40-392d-620b-8005-99636507e079"}})

{'topic': 'pasta',
 'joke': 'What kind of pasta does a ghost eat?\nBOO-lognese!',
 'explaination': 'The joke **"What kind of pasta does a ghost eat?"** followed by the punchline **"BOO-lognese!"** works because it plays on two things:\n\n1. **Homophones** – The word *"BOO-lognese"* sounds almost identical to *"Bolognese"* (the classic Italian pasta sauce made with meat and tomatoes).\n2. **Ghostly Wordplay** – The joke starts with *"BOO,"* a common sound or word associated with ghosts (like *"boo!"* when you scare someone). By replacing the *"B"* in *"Bolognese"* with *"BOO,"* the punchline humorously suggests that ghosts eat a spooky version of the dish.\n\nSo, the humor comes from the unexpected twist of turning a familiar food into something ghostly through wordplay! 👻🍝'}